# M9.2 · Yoğunluk dönüşümleri ve histogram eşitleme

Tek piksel işlemleri: negatif, log, gamma, parçalı doğrusal germe ve en önemlisi histogram eşitleme. 8 seviyeli klasik örnekte eşitleme dönüşümünü elle çıkarıyor, saf Python'da derste anlatılan ve OpenCV'nin kullandığı iki eşitleme formülünü yazıp OpenCV ile birebir karşılaştırıyor, NumPy'da tüm dönüşümleri arama tablosu (LUT) olarak döngüsüz kuruyor, PyTorch'ta türevlenebilir "yumuşak histogram" ile ton eğrisini gradient descent ile öğreniyor ve en sonda yüzlerce görüntüyü tek bir `bincount` ile eşitliyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-2-intensity-histogram.html](https://m-gonul.github.io/pytorch-lab/m9-2-intensity-histogram.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_2_intensity_histogram.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"cv2/numpy kullanılmıyor\", lambda: no_imports(src, {\"cv2\", \"numpy\", \"skimage\", \"scipy\"}))\n    R.test(\"histogram\", lambda: ensure(list(need(ns, \"histogram\")([[0, 0, 1], [255, 1, 1]]))[:2] == [2, 3] and need(ns, \"histogram\")([[0, 0, 1], [255, 1, 1]])[255] == 1, \"sayımlar\"))\n    crops = {\"camera\": data.camera()[100:228, 100:228], \"moon\": data.moon()[200:328, 200:328]}\n\n    def ocv():\n        for name, im in crops.items():\n            got = np.array(need(ns, \"equalize_opencv\")(im.tolist()), np.uint8)\n            ensure(np.array_equal(got, cv2.equalizeHist(im)), f\"{name}: cv2.equalizeHist ile farklı ({int(np.abs(got.astype(int) - cv2.equalizeHist(im)).sum())})\")\n        ensure(need(ns, \"equalize_opencv\")([[7, 7], [7, 7]]) == [[7, 7], [7, 7]], \"tek renkli görüntü değişmemeli\")\n    R.test(\"equalize_opencv = cv2.equalizeHist (birebir)\", ocv)\n\n    def cls():\n        rows = []\n        for name, im in crops.items():\n            got = np.array(need(ns, \"equalize_class\")(im.tolist()), np.int64)\n            c = np.bincount(im.ravel(), minlength=256).cumsum()\n            ensure(np.array_equal(got, np.round(255 * c / im.size).astype(np.int64)[im]), f\"{name}: ders formülü\")\n            rows.append([name, str(int(np.abs(got - cv2.equalizeHist(im)).sum()))])\n        R.table(\"Toplam mutlak fark: ders formülü − OpenCV\", [\"görüntü\", \"Σ|fark|\"], rows,\n                note=\"moon'da en koyu seviyede çok piksel var: c_min büyük, iki formül ayrışıyor.\")\n    R.test(\"equalize_class (ders formülü)\", cls)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    ims = {\"camera\": data.camera(), \"moon\": data.moon(), \"coins\": data.coins(), \"page\": data.page()}\n\n    def eq():\n        for name, im in ims.items():\n            got = need(ns, \"equalize\")(im, \"opencv\")\n            ensure(got.dtype == np.uint8 and np.array_equal(got, cv2.equalizeHist(im)), f\"{name}: cv2 ile farklı\")\n    R.test(\"equalize('opencv') = cv2.equalizeHist (4 görüntü)\", eq)\n\n    def eqc():\n        im = ims[\"moon\"]\n        c = np.bincount(im.ravel(), minlength=256).cumsum()\n        ensure(np.array_equal(need(ns, \"equalize\")(im, \"class\"), np.rint(255 * c / im.size).astype(np.uint8)[im]), \"ders formülü\")\n    R.test(\"equalize('class')\", eqc)\n\n    def pts():\n        im = ims[\"camera\"]\n        ensure(np.array_equal(need(ns, \"negative\")(im), 255 - im), \"negatif\")\n        r = np.arange(256, dtype=np.uint8).reshape(16, 16)\n        close(need(ns, \"log_transform\")(r).astype(float), np.rint(255 / np.log(256) * np.log1p(r.astype(float))), what=\"log\")\n        close(need(ns, \"gamma\")(r, 0.5).astype(float), np.rint(255 * (r / 255) ** 0.5), what=\"gamma 0.5\")\n        ensure(need(ns, \"gamma\")(r, 1.0).tolist() == r.tolist(), \"gamma 1 özdeşlik\")\n        g2 = ns[\"gamma\"](im, 2.2)\n        ensure(g2.dtype == np.uint8 and g2.mean() < im.mean(), \"γ > 1 koyulaştırır\")\n    R.test(\"negatif, log, gamma\", pts)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef entropy(img_u8, bins=16):\n    p = np.bincount(((img_u8.astype(np.int64) * bins) // 256).ravel(), minlength=bins) / img_u8.size\n    p = p[p > 0]\n    return float(-(p * np.log2(p)).sum())\n\n\ndef check(ns, R):\n    x = torch.tensor(data.moon(), dtype=torch.float32) / 255\n\n    def sh():\n        v = torch.tensor([0.15, 0.15, 0.95], requires_grad=True)\n        h = need(ns, \"soft_histogram\")(v, bins=10)\n        close(h.sum(), 1.0)\n        ensure(h.argmax().item() == 1 and h[1] > h[9] > h[5], \"tepe 2. kutuda, ikinci tepe sonuncuda\")\n        (h[1]).backward()\n        ensure(v.grad is not None and v.grad.abs().sum() > 0, \"gradyan akmalı\")\n    R.test(\"soft_histogram: normalize ve türevlenebilir\", sh)\n\n    def tc():\n        c = need(ns, \"ToneCurve\", \"sınıf\")()\n        y = c(torch.linspace(0, 1, 101))\n        close(y[0], 0.0, atol=1e-6); close(y[-1], 1.0, atol=1e-6)\n        close(y, torch.linspace(0, 1, 101), atol=1e-5, what=\"başlangıçta özdeşlik\")\n    R.test(\"ToneCurve: başlangıçta özdeşlik, uçlar sabit\", tc)\n\n    def learn():\n        torch.manual_seed(0)\n        curve = need(ns, \"learn_equalization\")(x)\n        with torch.no_grad():\n            grid = curve(torch.linspace(0, 1, 256))\n            out = (curve(x) * 255).round().clamp(0, 255).to(torch.uint8).numpy()\n        ensure(bool((grid[1:] >= grid[:-1] - 1e-7).all()), \"eğri monoton olmalı\")\n        e0, e1, e2 = entropy(data.moon()), entropy(out), entropy(cv2.equalizeHist(data.moon()))\n        ensure(e1 > e0 + 1.0 and e1 > 0.95 * e2, f\"16 kutulu entropi: orijinal {e0:.2f}, öğrenilen {e1:.2f}, cv2 HE {e2:.2f} bit\")\n        return f\"16 kutulu histogram entropisi (bit, en fazla 4): orijinal {e0:.2f} → öğrenilen eğri {e1:.2f} (cv2 HE {e2:.2f})\"\n    R.test(\"öğrenilen ton eğrisi histogramı düzleştiriyor\", learn)\n",
 "e": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: max_loops(src, 0, func=\"equalize_batch\"))\n\n    def eq():\n        ims = [data.camera(), data.moon(), data.coins()[:303, :384][:256, :256], data.page()[:172, :256]]\n        ims = [cv2.resize(i, (256, 256)) for i in ims] + [np.full((256, 256), 9, np.uint8)]\n        X = torch.tensor(np.stack(ims))\n        got = need(ns, \"equalize_batch\")(X).numpy()\n        for i, im in enumerate(ims):\n            ensure(np.array_equal(got[i], cv2.equalizeHist(im)), f\"görüntü {i} cv2 ile farklı\")\n    R.test(\"her görüntü = cv2.equalizeHist (tek renkli dahil)\", eq)\n    if \"equalize_batch\" not in ns:\n        return\n    g = torch.Generator().manual_seed(0)\n    X = (torch.rand(512, 128, 128, generator=g) ** 2 * 255).to(torch.uint8)\n    Xn = X.numpy()\n    t_loop = bench(lambda: [cv2.equalizeHist(Xn[i]) for i in range(512)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"equalize_batch\"](X), repeat=3)\n    R.table(\"512 görüntü, 128 × 128\", [\"yöntem\", \"süre\"], [[\"cv2.equalizeHist döngüsü\", fmt_time(t_loop)], [\"tek bincount (torch)\", fmt_time(t_vec)]],\n            note=\"OpenCV'nin tek görüntülük C++ çekirdeği çok hızlı; batch'li sürüm GPU'da ve veri artırma hattında anlam kazanır.\")\n",
 "x1": "import numpy as np\nfrom skimage import data, exposure\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    for a, b in ((\"moon\", \"camera\"), (\"camera\", \"coins\"), (\"page\", \"moon\")):\n        def t(a=a, b=b):\n            s, r = getattr(data, a)(), getattr(data, b)()\n            close(need(ns, \"match_histograms\")(s, r), exposure.match_histograms(s, r), atol=1e-9)\n        R.test(f\"{a} → {b} = skimage\", t)\n",
 "x2": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n\n    def t():\n        f = img.astype(np.float64)\n        m = ndimage.uniform_filter(f, 3, mode=\"mirror\")\n        s = np.sqrt(np.maximum(ndimage.uniform_filter(f * f, 3, mode=\"mirror\") - m * m, 0))\n        sel = (m <= 0.4 * f.mean()) & (s >= 0.02 * f.std()) & (s <= 0.4 * f.std())\n        want = np.clip(np.where(sel, 4 * f, f), 0, 255).astype(np.uint8)\n        got = need(ns, \"local_enhance\")(img)\n        diff = np.abs(got.astype(int) - want.astype(int))\n        ensure(got.dtype == np.uint8 and (diff > 1).mean() < 1e-3, f\"{(diff > 1).mean():.2%} piksel farklı\")\n        return f\"seçilen piksel oranı {sel.mean():.1%}\"\n    R.test(\"local_enhance (Gonzalez yerel istatistik kuralı)\", t)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.moon()\n\n    def st():\n        got = need(ns, \"stretch\")(img)\n        ref = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)\n        ensure(got.dtype == np.uint8 and np.abs(got.astype(int) - ref.astype(int)).max() <= 1, \"cv2.normalize ile ±1\")\n        ensure(got.min() == 0 and got.max() == 255, \"tam aralık\")\n    R.test(\"stretch = cv2.normalize(NORM_MINMAX)\", st)\n\n    def pw():\n        r = np.arange(256, dtype=np.uint8)\n        y = need(ns, \"piecewise\")(r, 80, 20, 160, 235)\n        ensure(int(y[80]) == 20 and int(y[160]) == 235 and int(y[0]) == 0 and int(y[255]) == 255, \"düğüm noktaları\")\n        ensure(bool(np.all(np.diff(y.astype(int)) >= 0)), \"monoton\")\n        ensure(abs(int(y[120]) - round(20 + 215 * 40 / 80)) <= 0, \"orta parça doğrusal\")\n    R.test(\"piecewise\", pw)\n\n    def sl():\n        b = need(ns, \"slice_level\")(img, 100, 150, \"binary\")\n        ensure(set(np.unique(b)) <= {0, 255} and np.array_equal(b == 255, (img >= 100) & (img <= 150)), \"binary\")\n        p = ns[\"slice_level\"](img, 100, 150, \"preserve\")\n        ensure(np.array_equal(p[(img < 100)], img[img < 100]) and np.all(p[(img >= 100) & (img <= 150)] == 255), \"preserve\")\n    R.test(\"slice_level (iki mod)\", sl)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_2_intensity_histogram as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Nokta işlemi:** her çıkış pikseli yalnızca aynı konumdaki giriş pikseline bağlı, $s=T(r)$. 8 bitte $T$ 256 girişlik bir **arama tablosudur** (LUT): `out = lut[img]`.

| dönüşüm | $T(r)$ | etkisi |
|---|---|---|
| negatif | $L-1-r$ | koyu bölgelerdeki beyaz ayrıntıyı öne çıkarır |
| log | $c\log(1+r)$ | geniş dinamik aralığı sıkıştırır (Fourier spektrumu) |
| gamma | $c\,r^\gamma$ | $\gamma<1$ açar, $\gamma>1$ koyulaştırır |
| kontrast germe | parçalı doğrusal | seçilen aralığı genişletir |

**histogram eşitleme.** $n_k$ seviye $k$'deki piksel sayısı, $p_k=n_k/MN$. Sürekli durumda $s=(L-1)\int_0^rp(w)\,dw$ çıkış yoğunluğunu düzgün yapar. Ayrık hâli:

$$
s_k=\operatorname{round}\Big((L-1)\sum_{j=0}^{k}p_j\Big).
$$

OpenCV'nin `equalizeHist`'i en düşük dolu seviyenin birikimli sayısını ($c_{\min}$) çıkarır ve çıkışı tam aralığa yayar:

$$
s_k=\operatorname{round}\Big((L-1)\frac{c_k-c_{\min}}{MN-c_{\min}}\Big).
$$

İki koşul: $T$ monoton artan (yoğunluk sırası tersine dönmez) ve $0\le T(r)\le L-1$ (aralık korunur).

**Histogram eşleme:** girdinin CDF'sini hedef görüntünün CDF'sinin tersiyle birleştir: $s=G^{-1}(T(r))$.

## 1 · Elle

8 seviyeli örnekte dönüşüm yalnız 5 farklı çıkış seviyesi üretiyor: $r_3$ ve $r_4$ aynı 6'ya, $r_5,r_6,r_7$ aynı 7'ye gidiyor.

OpenCV varyantında en karanlık seviye her zaman 0'a gider; ders formülünde $7\cdot790/4096=1.35\to1$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**3 bit ($L=8$), $64\times64$ görüntü; histogram $n_k=(790,1023,850,656,329,245,122,81)$. Ders formülü $s_k=\operatorname{round}\big(7\sum_{j\le k}p_j\big)$**

- `s0`: $s_0$
- `s1`: $s_1$
- `s3`: $s_3$
- `n6`: Eşitlenmiş görüntüde 6 değerli piksel sayısı

**OpenCV varyantı $s_k=\operatorname{round}\big(7\,\frac{c_k-c_{\min}}{MN-c_{\min}}\big)$, $c_k$ birikimli sayım**

- `o0`: $s_0$
- `o1`: $s_1$

**Nokta dönüşümleri, 8 bit**

- `gam`: Gamma $\gamma=0.5$: $r=64$ → $255\,(64/255)^{0.5}$
- `lg`: Log: $c=255/\ln256$, $r=15$ → $c\ln(1+r)$

In [ ]:
cevaplar = dict(
    s0=None,
    s1=None,
    s3=None,
    n6=None,
    o0=None,
    o1=None,
    gam=None,
    lg=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"s0":[1,0],"s1":[3,0],"s3":[6,0],"n6":[985,0],"o0":[0,0],"o1":[2,0],"gam":[127.749755,0.01],"lg":[127.5,0.01]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; görüntü `uint8` değerlerinden oluşan liste listesi.

- `histogram(img)` → 256 elemanlı sayım listesi.
- `equalize_class(img)` — ders formülü, `round((L-1)·c_k/MN)`.
- `equalize_opencv(img)` — $c_{\min}$'li formül; $MN=c_{\min}$ (tek renkli görüntü) ise görüntüyü olduğu gibi döndür.

Kontrol `cv2.equalizeHist` ile birebir karşılaştırır ve iki formülün farkını (ödevdeki "toplam mutlak fark") raporlar.

In [ ]:
%%writefile stage_b.py
def histogram(img):
    ...


def equalize_class(img):
    ...


def equalize_opencv(img):
    ...


img = [[0, 0, 1, 1], [2, 2, 2, 3]]
print(histogram(img)[:4], equalize_class(img), equalize_opencv(img))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**; her dönüşüm önce 256 elemanlı bir LUT kurar, sonra `lut[img]`.

- `equalize(img, variant)` — `"class"` ya da `"opencv"`; `np.bincount(…, minlength=256)` ve `cumsum`.
- `negative(img)`, `log_transform(img)` ($c=255/\ln256$), `gamma(img, g)` ($255\,(r/255)^g$) — çıkışlar `uint8`'e yuvarlanır (`np.rint`).

In [ ]:
%%writefile stage_c.py
import numpy as np


def equalize(img, variant="opencv"):
    ...


def negative(img):
    ...


def log_transform(img):
    ...


def gamma(img, g):
    ...


from skimage import data
img = data.moon()
print(equalize(img).min(), equalize(img).max(), gamma(img, 0.5).mean())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Türevlenebilir histogram eşitleme.

- `soft_histogram(x, bins=64, sigma=None)` — `x` [0, 1] aralığında; her pikselin her kutuya katkısı $\exp(-(x-c_k)^2/2\sigma^2)$ (varsayılan $\sigma$ = kutu genişliği), normalize. Gradyan akar.
- `ToneCurve(n=64)` — monoton artan bir eğri: `softmax(logits)`'in birikimli toplamı, düğümler arası doğrusal interpolasyon; `forward(x)` [0,1] → [0,1].
- `learn_equalization(x, steps=300, lr=0.05)` → `ToneCurve`: çıkışın yumuşak histogramının **entropisini** büyüt (düzgün dağılım = en yüksek entropi).

Kontrol öğrenilen eğrinin monoton olduğunu ve çıkışın **16 kutulu** histogram entropisinin klasik HE'ninkine ulaştığını sınar. (256 seviyede entropi deterministik bir eşlemeyle artamaz: eşitleme seviyeleri birleştirir, ayıramaz. Düzlük kaba kutularda ölçülür.)

In [ ]:
%%writefile stage_d.py
import math
import torch


def soft_histogram(x, bins=64, sigma=None):
    ...


class ToneCurve(torch.nn.Module):
    def __init__(self, n=64):
        super().__init__()
        ...

    def forward(self, x):
        ...


def learn_equalization(x, steps=300, lr=0.05):
    ...


from skimage import data
x = torch.tensor(data.moon(), dtype=torch.float32) / 255
curve = learn_equalization(x)
print(curve(torch.linspace(0, 1, 5)))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`equalize_batch(imgs)` — `imgs: (B, H, W) uint8` tensör, OpenCV varyantı, **döngü yok**. Her görüntünün histogramını tek bir `bincount` ile al: piksel değerine görüntü indeksi × 256 ekle. CDF'ler `cumsum(dim=1)`, LUT'lar `gather`.

In [ ]:
%%writefile stage_e.py
import torch


def equalize_batch(imgs):
    """imgs: (B, H, W) uint8. OpenCV varyantı. Döngü yok."""
    ...


imgs = torch.randint(30, 120, (4, 64, 64), dtype=torch.uint8)
print(equalize_batch(imgs).float().mean((1, 2)))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Histogram eşleme (specification)

Histogram eşleme (specification): `src` görüntünün histogramını `ref` görüntüsününkine benzet.

- `match_histograms(src, ref)` — her benzersiz giriş değerinin nicelik (quantile) değerini hedefin nicelik–değer eğrisinde `np.interp` ile bul. `np.unique(…, return_inverse=True, return_counts=True)`. Döngü yok.

Kontrol `skimage.exposure.match_histograms` ile birebir karşılaştırır.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def match_histograms(src, ref):
    ...


from skimage import data
print(match_histograms(data.moon(), data.camera()).mean(), data.camera().mean())

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Yerel histogram istatistikleriyle iyileştirme

Global eşitleme koyu bir bölgedeki ayrıntıyı ortaya çıkaramayabilir. Yerel istatistiklerle iyileştirme: yerel ortalama $m_{S}$ ve standart sapma $\sigma_S$ ($k\times k$ pencere), global $m_G$, $\sigma_G$:

$$
g=\begin{cases}E\cdot f & m_S\le k_0m_G\ \text{ve}\ k_1\sigma_G\le\sigma_S\le k_2\sigma_G\\ f & \text{aksi}\end{cases}
$$

- `local_enhance(img, E=4.0, k0=0.4, k1=0.02, k2=0.4, size=3)` — yerel ortalama ve varyansı `cv2.blur` (BORDER_REFLECT_101) ile hesapla: $\sigma_S^2=\overline{f^2}-\bar f^2$. Çıkış `uint8`'e kırpılır.

In [ ]:
%%writefile extra_x2.py
import numpy as np
import cv2


def local_enhance(img, E=4.0, k0=0.4, k1=0.02, k2=0.4, size=3):
    ...


from skimage import data
print(local_enhance(data.camera()).mean())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Kontrast germe ve yoğunluk dilimleme

- `stretch(img)` — en küçük değeri 0'a, en büyüğü 255'e doğrusal ger (`cv2.normalize(…, 0, 255, cv2.NORM_MINMAX)` ile aynı, `np.rint`).
- `piecewise(img, r1, s1, r2, s2)` — $(0,0)$, $(r_1,s_1)$, $(r_2,s_2)$, $(255,255)$ noktalarından geçen parçalı doğrusal dönüşüm (`np.interp`).
- `slice_level(img, lo, hi, mode)` — $[lo, hi]$ aralığını vurgula: `"binary"` aralık 255, diğerleri 0; `"preserve"` aralık 255, diğerleri değişmez.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def stretch(img):
    ...


def piecewise(img, r1, s1, r2, s2):
    ...


def slice_level(img, lo, hi, mode="binary"):
    ...


from skimage import data
img = data.moon()
print(stretch(img).min(), stretch(img).max())

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Kendi tone-mapping hattın: HDR'den 8 bite

[Reinhard ve ark. — Photographic Tone Reproduction for Digital Images (SIGGRAPH 2002)](https://doi.org/10.1145/566654.566575)

Bu modülün nokta dönüşümlerini bir HDR görüntüyü (ör. `cv2.createMergeDebevec` ile birleştirilmiş çok pozlamalı seri ya da sentetik bir HDR sahne) 8 bite indiren bir hatta birleştir: log ortalama parlaklıkla ölçekleme, Reinhard global operatörü, gamma ve isteğe bağlı CLAHE (M9.12).


**Adımlar**

1. Log-ortalama parlaklık $\bar L_w=\exp(\operatorname{mean}\log(\delta+L_w))$ ve anahtar değer $a$ ile ölçekleme yaz.
2. Global operatör $L_d=L(1+L/L_{white}^2)/(1+L)$; renk kanallarını parlaklık oranıyla koru.
3. Aynı sahneye basit histogram eşitleme ve gamma uygula; sonuçları yan yana göster.
4. `cv2.createTonemapReinhard` ile karşılaştır.

**Kabul ölçütleri**

- Çıkış 8 bit ve kırpılan piksel oranı %1'in altında.
- Parlak ve karanlık bölgelerin ikisinde de ayrıntı görünüyor (her ikisinde yerel std > 5 gri seviye).
- OpenCV'nin Reinhard operatörüyle aynı parametrelerde ortalama mutlak fark < 10 gri seviye.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 3.2–3.3](https://www.imageprocessingplace.com/) — ©
- [OpenCV — Histogram Equalization tutorial ve equalizeHist](https://docs.opencv.org/4.x/d4/d1b/tutorial_histogram_equalization.html) — Apache-2.0
- [scikit-image — exposure modülü (equalize_hist, match_histograms)](https://scikit-image.org/docs/stable/api/skimage.exposure.html) — BSD-3-Clause